### Objetivo: Consumir API

Acessando a url https://dadosabertos.bcb.gov.br/dataset/estatisticas-meios-pagamentos vocês terão acesso aos dados fornecidos pelo banco central sobre meios de pagamentos.
Vocês podem consultar a API para obter a url para requisição dos dados.


Determine a url fornecida:

In [1]:
url = "https://olinda.bcb.gov.br/olinda/servico/Pix_DadosAbertos/versao/v1/odata/TransacoesPixPorMunicipio(DataBase=@DataBase)?@DataBase='202301'&$top=100&$format=json&$select=AnoMes,Municipio_Ibge,Municipio,Estado_Ibge,Estado,Sigla_Regiao,Regiao,VL_PagadorPF,QT_PagadorPF,VL_PagadorPJ,QT_PagadorPJ,VL_RecebedorPF,QT_RecebedorPF,VL_RecebedorPJ,QT_RecebedorPJ,QT_PES_PagadorPF,QT_PES_PagadorPJ,QT_PES_RecebedorPF,QT_PES_RecebedorPJ"

Importar bibliotecas

In [2]:
import requests  #possibilita integração com serviços web, solicitações HTTP consumir dados de APIs
import json  #manipular arquivo json

Criar uma função para obter os dados:

In [3]:

def requisicao_api(link):
  resposta = requests.get(link)

  if resposta.status_code == 200:      #sucesso na requisição
    dados = resposta.json()
    print('Status Code:', resposta.status_code)
    with open('resultado.json', 'w', encoding='utf-8') as arquivo:   #salvar arquivo json
      json.dump(dados, arquivo, ensure_ascii=False, indent=4)

  else:
    print('Status Code:', resposta.status_code)


Chamar função passando o link gerado pela interface da API

In [4]:
requisicao_api(url)

Status Code: 200


In [5]:
with open("resultado.json", "r", encoding="utf-8") as f:
    dados = json.load(f)

# Extrai apenas o conteúdo da chave "value"
registros = dados["value"]


In [6]:
print(registros)

[{'AnoMes': 202502, 'Municipio_Ibge': 2106375, 'Municipio': 'MARANHÃOZINHO', 'Estado_Ibge': 21, 'Estado': 'MARANHÃO', 'Sigla_Regiao': 'NE', 'Regiao': 'NORDESTE', 'VL_PagadorPF': 26792818.65, 'QT_PagadorPF': 207588, 'VL_PagadorPJ': 7540820.84, 'QT_PagadorPJ': 6119, 'VL_RecebedorPF': 24876379.46, 'QT_RecebedorPF': 122444, 'VL_RecebedorPJ': 6186546.82, 'QT_RecebedorPJ': 25816, 'QT_PES_PagadorPF': 5368, 'QT_PES_PagadorPJ': 105, 'QT_PES_RecebedorPF': 5180, 'QT_PES_RecebedorPJ': 99}, {'AnoMes': 202304, 'Municipio_Ibge': 5201207, 'Municipio': 'ANHANGUERA', 'Estado_Ibge': 52, 'Estado': 'GOIÁS', 'Sigla_Regiao': 'CO', 'Regiao': 'CENTRO-OESTE', 'VL_PagadorPF': 1708642.5, 'QT_PagadorPF': 7693, 'VL_PagadorPJ': 126968.35, 'QT_PagadorPJ': 321, 'VL_RecebedorPF': 1332932.68, 'QT_RecebedorPF': 5671, 'VL_RecebedorPJ': 184304.66, 'QT_RecebedorPJ': 626, 'QT_PES_PagadorPF': 445, 'QT_PES_PagadorPJ': 20, 'QT_PES_RecebedorPF': 441, 'QT_PES_RecebedorPJ': 19}, {'AnoMes': 202402, 'Municipio_Ibge': 2600401, 'Munic

###Objetivo: Processamento de dados com Spark

Instalando Java e Pyspark:

In [7]:
!apt-get update -qq > /dev/null
!apt-get install openjdk-11-jdk -qq > /dev/null
!pip install -q pyspark

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


Definir variáveis de ambiente para especificar onde o Spark está instalado:

In [8]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["PATH"] += os.pathsep + os.path.join(os.environ["JAVA_HOME"], "bin")

Importar Bibliotecas e iniciar seção:

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, round

# ---- Criar sessão Spark ----
spark = SparkSession.builder \
    .appName("BCB_API_PySpark") \
    .master("local[*]") \
    .getOrCreate()


Início ao processamento de dados:

In [10]:
df = spark.createDataFrame(registros)

# Mostra o esquema e os dados
df.printSchema()

root
 |-- AnoMes: long (nullable = true)
 |-- Estado: string (nullable = true)
 |-- Estado_Ibge: long (nullable = true)
 |-- Municipio: string (nullable = true)
 |-- Municipio_Ibge: long (nullable = true)
 |-- QT_PES_PagadorPF: long (nullable = true)
 |-- QT_PES_PagadorPJ: long (nullable = true)
 |-- QT_PES_RecebedorPF: long (nullable = true)
 |-- QT_PES_RecebedorPJ: long (nullable = true)
 |-- QT_PagadorPF: long (nullable = true)
 |-- QT_PagadorPJ: long (nullable = true)
 |-- QT_RecebedorPF: long (nullable = true)
 |-- QT_RecebedorPJ: long (nullable = true)
 |-- Regiao: string (nullable = true)
 |-- Sigla_Regiao: string (nullable = true)
 |-- VL_PagadorPF: double (nullable = true)
 |-- VL_PagadorPJ: double (nullable = true)
 |-- VL_RecebedorPF: double (nullable = true)
 |-- VL_RecebedorPJ: double (nullable = true)



In [11]:
df.show(5, truncate=False)

+------+----------+-----------+-------------+--------------+----------------+----------------+------------------+------------------+------------+------------+--------------+--------------+------------+------------+-------------+------------+--------------+--------------+
|AnoMes|Estado    |Estado_Ibge|Municipio    |Municipio_Ibge|QT_PES_PagadorPF|QT_PES_PagadorPJ|QT_PES_RecebedorPF|QT_PES_RecebedorPJ|QT_PagadorPF|QT_PagadorPJ|QT_RecebedorPF|QT_RecebedorPJ|Regiao      |Sigla_Regiao|VL_PagadorPF |VL_PagadorPJ|VL_RecebedorPF|VL_RecebedorPJ|
+------+----------+-----------+-------------+--------------+----------------+----------------+------------------+------------------+------------+------------+--------------+--------------+------------+------------+-------------+------------+--------------+--------------+
|202502|MARANHÃO  |21         |MARANHÃOZINHO|2106375       |5368            |105             |5180              |99                |207588      |6119        |122444        |25816      

**Transformações:** Criam novos DataFrames sem processar imediatamente os dados

In [14]:
from pyspark.sql import functions as F

In [12]:
# 1. Selecionar colunas específicas
df_select = df.select("QT_PES_PagadorPF", "QT_RecebedorPF")
df_select.show(5)

+----------------+--------------+
|QT_PES_PagadorPF|QT_RecebedorPF|
+----------------+--------------+
|            5368|        122444|
|             445|          5671|
|           12744|        246967|
|            2227|         38292|
|            2033|         20145|
+----------------+--------------+
only showing top 5 rows



In [15]:
# 2. Criar novas colunas
df_novo = df.withColumn("totalTransacoes", F.col("QT_PagadorPF") + F.col("QT_PagadorPJ") + F.col("QT_RecebedorPF") + F.col("QT_RecebedorPJ"))
df_novo.show(5)

+------+----------+-----------+-------------+--------------+----------------+----------------+------------------+------------------+------------+------------+--------------+--------------+------------+------------+-------------+------------+--------------+--------------+---------------+
|AnoMes|    Estado|Estado_Ibge|    Municipio|Municipio_Ibge|QT_PES_PagadorPF|QT_PES_PagadorPJ|QT_PES_RecebedorPF|QT_PES_RecebedorPJ|QT_PagadorPF|QT_PagadorPJ|QT_RecebedorPF|QT_RecebedorPJ|      Regiao|Sigla_Regiao| VL_PagadorPF|VL_PagadorPJ|VL_RecebedorPF|VL_RecebedorPJ|totalTransacoes|
+------+----------+-----------+-------------+--------------+----------------+----------------+------------------+------------------+------------+------------+--------------+--------------+------------+------------+-------------+------------+--------------+--------------+---------------+
|202502|  MARANHÃO|         21|MARANHÃOZINHO|       2106375|            5368|             105|              5180|                99|    

In [17]:
# 3. Filtrar registros (condição)
df_filtrado = df.filter(F.col("VL_PagadorPJ") < 1_000_000)
df_filtrado.show(5)

+------+------+-----------+--------------------+--------------+----------------+----------------+------------------+------------------+------------+------------+--------------+--------------+------------+------------+------------+------------+--------------+--------------+
|AnoMes|Estado|Estado_Ibge|           Municipio|Municipio_Ibge|QT_PES_PagadorPF|QT_PES_PagadorPJ|QT_PES_RecebedorPF|QT_PES_RecebedorPJ|QT_PagadorPF|QT_PagadorPJ|QT_RecebedorPF|QT_RecebedorPJ|      Regiao|Sigla_Regiao|VL_PagadorPF|VL_PagadorPJ|VL_RecebedorPF|VL_RecebedorPJ|
+------+------+-----------+--------------------+--------------+----------------+----------------+------------------+------------------+------------+------------+--------------+--------------+------------+------------+------------+------------+--------------+--------------+
|202304| GOIÁS|         52|          ANHANGUERA|       5201207|             445|              20|               441|                19|        7693|         321|          5671|  

In [19]:
# 4. Ordenar valores
df_ordenado = df.orderBy(F.col("QT_PES_PagadorPJ").asc())
df_ordenado.show(10)

+------+-----------------+-----------+--------------------+--------------+----------------+----------------+------------------+------------------+------------+------------+--------------+--------------+------------+------------+------------+------------+--------------+--------------+
|AnoMes|           Estado|Estado_Ibge|           Municipio|Municipio_Ibge|QT_PES_PagadorPF|QT_PES_PagadorPJ|QT_PES_RecebedorPF|QT_PES_RecebedorPJ|QT_PagadorPF|QT_PagadorPJ|QT_RecebedorPF|QT_RecebedorPJ|      Regiao|Sigla_Regiao|VL_PagadorPF|VL_PagadorPJ|VL_RecebedorPF|VL_RecebedorPJ|
+------+-----------------+-----------+--------------------+--------------+----------------+----------------+------------------+------------------+------------+------------+--------------+--------------+------------+------------+------------+------------+--------------+--------------+
|202304|            GOIÁS|         52|          ANHANGUERA|       5201207|             445|              20|               441|                19

In [21]:
# 5. Selecionar e renomear colunas
df_renomeado = df.select(
    F.col("Estado_Ibge").alias("Cod_Estado_IBGE"),
    F.col("Municipio_Ibge").alias("Cod_Municipio_IBGE"),
    F.col("QT_PES_PagadorPF").alias("Qtd_Pessoas_Pagadoras_PF"),
    F.col("QT_PES_PagadorPJ").alias("Qtd_Pessoas_Pagadoras_PJ"),
    F.col("QT_PES_RecebedorPF").alias("Qtd_Pessoas_Recebedoras_PF"),
    F.col("QT_PES_RecebedorPJ").alias("Qtd_Pessoas_Recebedoras_PJ"),
    F.col("VL_PagadorPF").alias("Valor_Pago_PF"),
    F.col("VL_PagadorPJ").alias("Valor_Pago_PJ"),
    F.col("VL_RecebedorPF").alias("Valor_Recebido_PF"),
    F.col("VL_RecebedorPJ").alias("Valor_Recebido_PJ")
)
df_renomeado.show(5)

+---------------+------------------+------------------------+------------------------+--------------------------+--------------------------+-------------+-------------+-----------------+-----------------+
|Cod_Estado_IBGE|Cod_Municipio_IBGE|Qtd_Pessoas_Pagadoras_PF|Qtd_Pessoas_Pagadoras_PJ|Qtd_Pessoas_Recebedoras_PF|Qtd_Pessoas_Recebedoras_PJ|Valor_Pago_PF|Valor_Pago_PJ|Valor_Recebido_PF|Valor_Recebido_PJ|
+---------------+------------------+------------------------+------------------------+--------------------------+--------------------------+-------------+-------------+-----------------+-----------------+
|             21|           2106375|                    5368|                     105|                      5180|                        99|2.679281865E7|   7540820.84|    2.487637946E7|       6186546.82|
|             52|           5201207|                     445|                      20|                       441|                        19|    1708642.5|    126968.35|       13329

In [31]:
# 6. Criar uma coluna categórica com base em condição
df_categoria = df.withColumn(
    "tamanho_municipio",
    F.when(
        (F.col("QT_PES_PagadorPF") + F.col("QT_PES_PagadorPJ") +
         F.col("QT_PES_RecebedorPF") + F.col("QT_PES_RecebedorPJ")) > 200_000,
        "Grande"
    ).when(
        (F.col("QT_PES_PagadorPF") + F.col("QT_PES_PagadorPJ") +
         F.col("QT_PES_RecebedorPF") + F.col("QT_PES_RecebedorPJ")) > 50_000,
        "Médio"
    ).otherwise("Pequeno")
)
df_categoria.show(5)

+------+----------+-----------+-------------+--------------+----------------+----------------+------------------+------------------+------------+------------+--------------+--------------+------------+------------+-------------+------------+--------------+--------------+-----------------+
|AnoMes|    Estado|Estado_Ibge|    Municipio|Municipio_Ibge|QT_PES_PagadorPF|QT_PES_PagadorPJ|QT_PES_RecebedorPF|QT_PES_RecebedorPJ|QT_PagadorPF|QT_PagadorPJ|QT_RecebedorPF|QT_RecebedorPJ|      Regiao|Sigla_Regiao| VL_PagadorPF|VL_PagadorPJ|VL_RecebedorPF|VL_RecebedorPJ|tamanho_municipio|
+------+----------+-----------+-------------+--------------+----------------+----------------+------------------+------------------+------------+------------+--------------+--------------+------------+------------+-------------+------------+--------------+--------------+-----------------+
|202502|  MARANHÃO|         21|MARANHÃOZINHO|       2106375|            5368|             105|              5180|                9

**Ações**: Executam o plano de transformação e retornam resultados

In [25]:
# 1. Mostrar registros
df.show(5)

+------+----------+-----------+-------------+--------------+----------------+----------------+------------------+------------------+------------+------------+--------------+--------------+------------+------------+-------------+------------+--------------+--------------+
|AnoMes|    Estado|Estado_Ibge|    Municipio|Municipio_Ibge|QT_PES_PagadorPF|QT_PES_PagadorPJ|QT_PES_RecebedorPF|QT_PES_RecebedorPJ|QT_PagadorPF|QT_PagadorPJ|QT_RecebedorPF|QT_RecebedorPJ|      Regiao|Sigla_Regiao| VL_PagadorPF|VL_PagadorPJ|VL_RecebedorPF|VL_RecebedorPJ|
+------+----------+-----------+-------------+--------------+----------------+----------------+------------------+------------------+------------+------------+--------------+--------------+------------+------------+-------------+------------+--------------+--------------+
|202502|  MARANHÃO|         21|MARANHÃOZINHO|       2106375|            5368|             105|              5180|                99|      207588|        6119|        122444|         25

In [26]:
# 2. Contar número total de registros
print("Total de registros:", df.count())

Total de registros: 100


In [27]:
# 3. Descrever estatísticas básicas
df.describe().show()

+-------+-----------------+---------+------------------+---------------+------------------+-----------------+-----------------+------------------+------------------+-------------------+------------------+-----------------+-----------------+------------+------------+--------------------+--------------------+--------------------+--------------------+
|summary|           AnoMes|   Estado|       Estado_Ibge|      Municipio|    Municipio_Ibge| QT_PES_PagadorPF| QT_PES_PagadorPJ|QT_PES_RecebedorPF|QT_PES_RecebedorPJ|       QT_PagadorPF|      QT_PagadorPJ|   QT_RecebedorPF|   QT_RecebedorPJ|      Regiao|Sigla_Regiao|        VL_PagadorPF|        VL_PagadorPJ|      VL_RecebedorPF|      VL_RecebedorPJ|
+-------+-----------------+---------+------------------+---------------+------------------+-----------------+-----------------+------------------+------------------+-------------------+------------------+-----------------+-----------------+------------+------------+--------------------+-----------

In [28]:
# 4. Calcular médias de colunas
df.select(
    F.mean("VL_PagadorPF").alias("media_valor_pagador_pf"),
    F.mean("VL_PagadorPJ").alias("media_valor_pagador_pj"),
    F.mean("VL_RecebedorPF").alias("media_valor_recebedor_pf"),
    F.mean("VL_RecebedorPJ").alias("media_valor_recebedor_pj"),
).show()

+----------------------+----------------------+------------------------+------------------------+
|media_valor_pagador_pf|media_valor_pagador_pj|media_valor_recebedor_pf|media_valor_recebedor_pj|
+----------------------+----------------------+------------------------+------------------------+
|      4.897756990575E8|  1.1797935417950006E9|    4.9362030718990004E8|    1.0671580690760999E9|
+----------------------+----------------------+------------------------+------------------------+



In [29]:
# 5. Obter o registro com maior valorPix
df.orderBy(F.col("VL_RecebedorPJ").desc()).limit(6).show()
df.orderBy(F.col("VL_RecebedorPF").desc()).limit(6).show()

+------+--------------+-----------+--------------+--------------+----------------+----------------+------------------+------------------+------------+------------+--------------+--------------+-------+------------+-----------------+-----------------+-----------------+-----------------+
|AnoMes|        Estado|Estado_Ibge|     Municipio|Municipio_Ibge|QT_PES_PagadorPF|QT_PES_PagadorPJ|QT_PES_RecebedorPF|QT_PES_RecebedorPJ|QT_PagadorPF|QT_PagadorPJ|QT_RecebedorPF|QT_RecebedorPJ| Regiao|Sigla_Regiao|     VL_PagadorPF|     VL_PagadorPJ|   VL_RecebedorPF|   VL_RecebedorPJ|
+------+--------------+-----------+--------------+--------------+----------------+----------------+------------------+------------------+------------+------------+--------------+--------------+-------+------------+-----------------+-----------------+-----------------+-----------------+
|202410|RIO DE JANEIRO|         33|RIO DE JANEIRO|       3304557|         4694474|          337402|           4587911|            311858|  

In [32]:
# 6. Calcular soma total por uma categoria criada
df_categoria.groupBy("tamanho_municipio").agg(
    F.sum("VL_PagadorPF").alias("soma_pagador_pf"),
    F.sum("VL_PagadorPJ").alias("soma_pagador_pj"),
    F.sum("VL_RecebedorPF").alias("soma_recebedor_pf"),
    F.sum("VL_RecebedorPJ").alias("soma_recebedor_pj")
).show()

+-----------------+--------------------+--------------------+--------------------+--------------------+
|tamanho_municipio|     soma_pagador_pf|     soma_pagador_pj|   soma_recebedor_pf|   soma_recebedor_pj|
+-----------------+--------------------+--------------------+--------------------+--------------------+
|          Pequeno|     2.36993047234E9|     1.40257549446E9|2.3564917541400003E9|     1.33548113354E9|
|            Médio|     5.84776881242E9|     4.23304099161E9|     5.93301803423E9|4.1484429166400003E9|
|           Grande|4.075987062099000...|1.123437376934300...|   4.107252093062E10|1.012318828574300...|
+-----------------+--------------------+--------------------+--------------------+--------------------+

